# RPI, Schedule Strength, and Program Finances

Two team-context datasets, both Division I and both keyed to the registry.

**RPI is Warren Nolan's computation** from public game results, not an official
NCAA statistic. Coverage is 2021-2026; earlier years are not published, so
functions return `None` rather than raising.

**Program finances** come from the federal EADA survey — public-domain data
every Title IV institution must file.

Covers: `rpi`, `rpi_rank`, `rpi_record`, `rpi_table`, `strength_of_schedule`,
`quadrant_record`, `home_road_neutral`, `nonconference_profile`,
`rpi_over_years`, `best_wins`, `program_finance`, `budget_percentile`,
`roster_size`, `coaching_staff_size`, `richest_programs`,
`conference_spending`, `finance_vs_rpi`

In [1]:
from ncaa_bbStats import *

## RPI and schedule strength

In [2]:
# Stored as ranks, where 1 is best
print("Tennessee 2024 RPI rank:", rpi_rank("Tennessee", 2024))
print("  strength of schedule :", strength_of_schedule("Tennessee", 2024))

# `rpi` is an alias for `rpi_rank`
print("  via the alias        :", rpi("Tennessee", 2024))

# Outside 2021-2026 there is simply no data
print("\n2010 (not published):", rpi_rank("Tennessee", 2010))

Tennessee 2024 RPI rank: 1
  strength of schedule : 12
  via the alias        : 1

2010 (not published): None


In [3]:
import json
record = rpi_record("Tennessee", 2024)
print(json.dumps({k: record[k] for k in list(record)[:14]}, indent=2))

{
  "team_id": "IPEDS:221759",
  "team_name": "Tennessee",
  "season": 2024,
  "conference": "SEC",
  "rpi_rank": 1,
  "sos_rank": 12,
  "nonconference_rpi_rank": 10,
  "nonconference_sos_rank": 73,
  "conference_wins": 22,
  "conference_losses": 8,
  "conference_win_pct": 0.7333,
  "overall_wins": 60,
  "overall_losses": 13,
  "overall_win_pct": 0.8219
}


In [4]:
# Quadrants group opponents by strength; Q1 is the toughest
for quadrant in (1, 2, 3, 4):
    result = quadrant_record("Tennessee", 2024, quadrant)
    print(f"  Q{quadrant}: {result['wins']:>2}-{result['losses']:<2} "
          f"({result['win_pct']:.3f})")

  Q1: 26-10 (0.722)
  Q2:  9-1  (0.900)
  Q3:  5-1  (0.833)
  Q4: 20-1  (0.952)


In [5]:
splits = home_road_neutral("Tennessee", 2024)
for venue, result in splits.items():
    print(f"  {venue:8s} {result['wins']:>2}-{result['losses']:<2} "
          f"({result['win_pct']:.3f})")

print("\nnon-conference:", nonconference_profile("Tennessee", 2024))

  home     40-4  (0.909)
  road      9-6  (0.600)
  neutral  11-3  (0.786)

non-conference: {'wins': 24, 'losses': 2, 'win_pct': 0.9231, 'rpi_rank': 10, 'sos_rank': 73}


In [6]:
print("Top of the 2025 RPI standings:")
for row in rpi_table(2025, n=10):
    print(f"  {row['rpi_rank']:>3}. {row['team_name']:22s} {row['conference']:12s} "
          f"{row['overall_wins']:>2}-{row['overall_losses']:<2}")

Top of the 2025 RPI standings:
    1. Arkansas               SEC          50-15
    2. Coastal Carolina       Sun Belt     56-13
    3. Vanderbilt             SEC          43-18
    4. LSU                    SEC          53-15
    5. Auburn                 SEC          41-20
    6. Texas                  SEC          44-14
    7. Georgia                SEC          43-17
    8. Oregon State           Independent  48-16
    9. North Carolina         ACC          46-15
   10. UCLA                   Big Ten      48-18


In [7]:
print("SEC by RPI, 2025:")
for row in rpi_table(2025, conference="SEC"):
    print(f"  {row['rpi_rank']:>3}. {row['team_name']:22s} "
          f"{row['overall_wins']:>2}-{row['overall_losses']:<2}")

SEC by RPI, 2025:
    1. Arkansas               50-15
    3. Vanderbilt             43-18
    4. LSU                    53-15
    5. Auburn                 41-20
    6. Texas                  44-14
    7. Georgia                43-17
   12. Ole Miss               43-21
   14. Tennessee              46-19
   15. Alabama                41-18
   17. Florida                39-22
   24. Oklahoma               38-22
   33. Mississippi State      36-23
   36. Kentucky               31-26
   50. Texas A&M              30-26
   75. South Carolina         28-29
  149. Missouri               16-39


In [8]:
print("Most Quadrant 1 wins in 2025 -- the best wins against good teams:")
for row in best_wins(2025, n=8):
    print(f"  {row['team_name']:22s} {row['conference']:12s} "
          f"Q1 {row['q1_wins']:>2}-{row['q1_losses']:<2}  RPI {row['rpi_rank']}")

Most Quadrant 1 wins in 2025 -- the best wins against good teams:
  Ole Miss               SEC          Q1 20-15  RPI 12
  LSU                    SEC          Q1 19-11  RPI 4
  Vanderbilt             SEC          Q1 18-13  RPI 3
  Arkansas               SEC          Q1 17-12  RPI 1
  Texas                  SEC          Q1 17-13  RPI 6
  Alabama                SEC          Q1 17-15  RPI 15
  Auburn                 SEC          Q1 16-14  RPI 5
  Louisville             ACC          Q1 15-16  RPI 27


In [9]:
print("Coastal Carolina, season by season:")
for row in rpi_over_years("Coastal Carolina"):
    print(f"  {row['season']}  RPI {row['rpi_rank']:>3}  SOS {row['sos_rank']:>3}  "
          f"{row['wins']:>2}-{row['losses']:<2}  {row['conference']}")

Coastal Carolina, season by season:
  2021  RPI  96  SOS  74  27-24  Sun Belt
  2022  RPI  29  SOS  43  39-20  Sun Belt
  2023  RPI  19  SOS  19  42-21  Sun Belt
  2024  RPI  34  SOS  21  36-25  Sun Belt
  2025  RPI   2  SOS  43  56-13  Sun Belt
  2026  RPI  30  SOS  30  37-23  Sun Belt


## Program finances

Budget figures are **percentiles within a reporting year**, not dollars.
Baseball budgets inflate a few percent annually, so raw figures are not
comparable across seasons; a percentile is.

In [10]:
import json
print(json.dumps(program_finance("Tennessee", 2025), indent=2))

{
  "team_id": "IPEDS:221759",
  "institution_name": "The University of Tennessee-Knoxville",
  "unitid": "221759",
  "season": 2025,
  "eada_year": 2025,
  "carried_forward": false,
  "state": "TN",
  "budget_pct": 1.0,
  "log_budget": 16.253133,
  "opex_per_player_pct": 0.985629,
  "log_opex_per_player": 10.414003,
  "log_budget_per_player": 12.245804,
  "roster_size": 55.0,
  "log_revenue": 16.144406,
  "net_revenue": -1179200.0,
  "coaching_staff_size": 4.0,
  "dept_recruiting_pct": 0.994611,
  "log_dept_recruiting": 15.133051,
  "log_dept_coach_salary": 15.06445
}


In [11]:
print("Tennessee 2025 budget percentile:", budget_percentile("Tennessee", 2025))
print("  roster size    :", roster_size("Tennessee", 2025))
print("  coaching staff :", coaching_staff_size("Tennessee", 2025))

Tennessee 2025 budget percentile: 1.0
  roster size    : 55.0
  coaching staff : 4.0


### The 2026 caveat

Institutions file the 2025-26 survey in October 2026, so **2026 carries 2025
forward**. Every such row says so, because a carried-forward figure quietly
treated as current is how wrong conclusions get published.

In [12]:
for season in (2025, 2026):
    row = program_finance("Tennessee", season)
    print(f"  season {season}: eada_year={row['eada_year']}, "
          f"carried_forward={row['carried_forward']}")

  season 2025: eada_year=2025, carried_forward=False
  season 2026: eada_year=2025, carried_forward=True


In [13]:
print("Highest-spending Division I baseball programs, 2025:")
for row in richest_programs(2025, n=10, division=1):
    print(f"  {row['budget_pct']:.3f}  {row['institution_name'][:44]:44s} "
          f"roster {row['roster_size']:.0f}")

Highest-spending Division I baseball programs, 2025:


  1.000  The University of Tennessee-Knoxville        roster 55
  0.999  Louisiana State University and Agricultural  roster 41
  0.998  Vanderbilt University                        roster 46
  0.998  University of Arkansas                       roster 43
  0.997  Texas A&M University-College Station         roster 48
  0.996  North Carolina State University at Raleigh   roster 39
  0.996  Texas Christian University                   roster 41
  0.995  University of South Carolina-Columbia        roster 43
  0.994  Clemson University                           roster 40
  0.993  Mississippi State University                 roster 41


In [14]:
print("Median baseball budget percentile by conference, 2025:")
for row in conference_spending(2025)[:10]:
    print(f"  {row['conference']:16s} {row['median_budget_pct']:.4f}  "
          f"({row['programs']} programs)")

Median baseball budget percentile by conference, 2025:


  SEC              0.9919  (16 programs)
  DI Independent   0.9856  (1 programs)
  ACC              0.9808  (16 programs)
  Big 12           0.9760  (14 programs)
  Big Ten          0.9725  (17 programs)
  The American     0.9491  (10 programs)
  WCC              0.9449  (9 programs)
  Mountain West    0.9383  (7 programs)
  Sun Belt         0.9335  (14 programs)
  Big West         0.9311  (11 programs)


In [15]:
# Spending set against results, ready for correlation work
rows = finance_vs_rpi(2025)
print(f"{len(rows)} programs with both a budget percentile and an RPI rank\n")

import statistics
print("budget percentile vs RPI rank correlation:",
      round(statistics.correlation([r["budget_pct"] for r in rows],
                                   [r["rpi_rank"] for r in rows]), 3),
      "\n(negative is expected: a better rank is a smaller number)")

print("\nTop 10 by RPI:")
for row in rows[:10]:
    print(f"  RPI {row['rpi_rank']:>3}  budget {row['budget_pct']:.3f}  "
          f"{row['institution_name'][:40]}")

296 programs with both a budget percentile and an RPI rank

budget percentile vs RPI rank correlation: -0.518 
(negative is expected: a better rank is a smaller number)

Top 10 by RPI:
  RPI   1  budget 0.998  University of Arkansas
  RPI   2  budget 0.983  Coastal Carolina University
  RPI   3  budget 0.998  Vanderbilt University
  RPI   4  budget 0.999  Louisiana State University and Agricultu
  RPI   5  budget 0.993  Auburn University
  RPI   6  budget 0.381  Texas College
  RPI   7  budget 0.989  University of Georgia
  RPI   8  budget 0.986  Oregon State University
  RPI   9  budget 0.987  University of North Carolina at Chapel H
  RPI  10  budget 0.983  University of California-Los Angeles
